In [9]:
import numpy as np
import re
from collections import Counter


# ===============================
# 1. Preprocess
# ===============================
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    return tokens


# ===============================
# 2. Build Vocabulary
# ===============================
def build_vocab(tokens):
    vocab = sorted(set(tokens))
    word2idx = {w: i for i, w in enumerate(vocab)}
    idx2word = {i: w for w, i in word2idx.items()}
    return vocab, word2idx, idx2word


# ===============================
# 3. Create Training Data (Skip-gram)
# ===============================
def create_data(tokens, window_size=2):
    pairs = []
    for i, word in enumerate(tokens):
        for j in range(-window_size, window_size + 1):
            if j != 0 and 0 <= i + j < len(tokens):
                pairs.append((word, tokens[i + j]))
    return pairs


# ===============================
# 4. One-hot Encoding
# ===============================
def one_hot(word, word2idx, vocab_size):
    vec = np.zeros(vocab_size)
    vec[word2idx[word]] = 1
    return vec


# ===============================
# 5. Training
# ===============================
def train(pairs, word2idx, vocab_size, embedding_dim=10, epochs=100, lr=0.01):

    # Initialize weights
    W1 = np.random.randn(vocab_size, embedding_dim)
    W2 = np.random.randn(embedding_dim, vocab_size)

    for _ in range(epochs):
        loss = 0

        for target, context in pairs:
            x = one_hot(target, word2idx, vocab_size)
            y = one_hot(context, word2idx, vocab_size)

            # Forward pass
            h = np.dot(W1.T, x)
            u = np.dot(W2.T, h)
            y_pred = softmax(u)

            # Loss (cross-entropy)
            loss += -np.sum(y * np.log(y_pred + 1e-9))

            # Backprop
            e = y_pred - y
            dW2 = np.outer(h, e)
            dW1 = np.outer(x, np.dot(W2, e))

            # Update
            W1 -= lr * dW1
            W2 -= lr * dW2

    return W1, W2


# ===============================
# 6. Softmax
# ===============================
def softmax(x):
    exp_x = np.exp(x - np.max(x))
    return exp_x / np.sum(exp_x)


# ===============================
# 7. Get Embeddings
# ===============================
def get_embeddings(W1, vocab):
    embeddings = {}
    for i, word in enumerate(vocab):
        embeddings[word] = W1[i]
    return embeddings


# ===============================
# 8. Cosine Similarity
# ===============================
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


# ===============================
# 9. Similar Words
# ===============================
def similar_words(word, embeddings, top_n=5):
    if word not in embeddings:
        return f"'{word}' not in vocabulary"

    similarities = {}
    for other_word in embeddings:
        if other_word != word:
            sim = cosine_similarity(embeddings[word], embeddings[other_word])
            similarities[other_word] = sim

    sorted_words = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
    return sorted_words[:top_n]


# ===============================
# 10. Main
# ===============================
def main():

    corpus = """
    It was the best of times, it was the worst of times,
    it was the age of wisdom, it was the age of foolishness,
    it was the epoch of belief, it was the epoch of incredulity,
    it was the season of light, it was the season of darkness,
    it was the spring of hope, it was the winter of despair.
    """

    # Preprocess
    tokens = preprocess(corpus)
    print("Tokens:", tokens)

    # Vocabulary
    vocab, word2idx, idx2word = build_vocab(tokens)
    vocab_size = len(vocab)

    # Data
    pairs = create_data(tokens)

    # Train
    W1, W2 = train(pairs, word2idx, vocab_size)

    # Embeddings
    embeddings = get_embeddings(W1, vocab)

    # Test word (SAFE)
    test_word = "hope"

    print("\nEmbedding Example:")
    if test_word in embeddings:
        print(f"{test_word} →", embeddings[test_word])
    else:
        print(f"{test_word} not found")

    print(f"\nSimilar to '{test_word}':")
    print(similar_words(test_word, embeddings))


# ===============================
# Run
# ===============================
if __name__ == "__main__":
    main()

Tokens: ['it', 'was', 'the', 'best', 'of', 'times', 'it', 'was', 'the', 'worst', 'of', 'times', 'it', 'was', 'the', 'age', 'of', 'wisdom', 'it', 'was', 'the', 'age', 'of', 'foolishness', 'it', 'was', 'the', 'epoch', 'of', 'belief', 'it', 'was', 'the', 'epoch', 'of', 'incredulity', 'it', 'was', 'the', 'season', 'of', 'light', 'it', 'was', 'the', 'season', 'of', 'darkness', 'it', 'was', 'the', 'spring', 'of', 'hope', 'it', 'was', 'the', 'winter', 'of', 'despair']

Embedding Example:
hope → [ 0.79863052  1.5318468  -0.2718641  -2.71818699  0.55913085  0.84713191
  0.27767996 -1.03759345 -0.29515708 -0.1604583 ]

Similar to 'hope':
[('age', np.float64(0.5498119345940613)), ('foolishness', np.float64(0.4125555481595786)), ('winter', np.float64(0.39036029946234435)), ('the', np.float64(0.35368463842618525)), ('darkness', np.float64(0.27152382823220095))]
